In [1]:
import os 
import torch
# torch.autograd.set_detect_anomaly(True)


# CLASSES_FILE = r'data\archive\class_names.txt'
CLASSES_FILE = r'C:\Users\Sistem Cerdas Five\Desktop\vian\afre\lesgo\LoLI-Street Dataset\YOLO Annotations\Train\high\Classes\class_names.txt'
BATCH_SIZE=8 
IMG_SIZE=(640, 640)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
from modules.data_obj import LoLiStreetDataset_Curated
from torch.utils.data import DataLoader
import torchvision.transforms as T
import pandas as pd 


# train_root = r'C:\Users\vian8\Desktop\Tugas2\LLIE\LoLI-Street Dataset\LoLI-Street Dataset\Train'
# val_root = r'C:\Users\vian8\Desktop\Tugas2\LLIE\LoLI-Street Dataset\LoLI-Street Dataset\Val'

train_root = r'C:\Users\Sistem Cerdas Five\Desktop\vian\afre\lesgo\LoLI-Street Dataset\Train'
val_root = r'C:\Users\Sistem Cerdas Five\Desktop\vian\afre\lesgo\LoLI-Street Dataset\Val'

# train_GT_root = r'C:\Users\vian8\Desktop\Tugas2\LLIE\LoLI-Street Dataset\LoLI-Street Dataset\YOLO Annotations\Train'
# val_GT_root = r'C:\Users\vian8\Desktop\Tugas2\LLIE\LoLI-Street Dataset\LoLI-Street Dataset\YOLO Annotations\Val'

train_GT_root = r'C:\Users\Sistem Cerdas Five\Desktop\vian\afre\lesgo\LoLI-Street Dataset\YOLO Annotations\Train'
val_GT_root = r'C:\Users\Sistem Cerdas Five\Desktop\vian\afre\lesgo\LoLI-Street Dataset\YOLO Annotations\Val'

# train_curated_pth = r'C:\Users\vian8\Desktop\Tugas2\LLIE\LoLI-Street Dataset\LoLI-Street Dataset\curated_dataset.csv'
# val_curated_pth = r'C:\Users\vian8\Desktop\Tugas2\LLIE\LoLI-Street Dataset\LoLI-Street Dataset\val_curated_dataset.csv'

train_curated_pth = r'C:\Users\Sistem Cerdas Five\Desktop\vian\afre\lesgo\New folder\curated_dataset.csv'
val_curated_pth = r'C:\Users\Sistem Cerdas Five\Desktop\vian\afre\lesgo\New folder\val_curated_dataset.csv'

# Define paths for the new split files
# val_split_pth = r'C:\Users\vian8\Desktop\Tugas2\LLIE\LoLI-Street Dataset\LoLI-Street Dataset\val_split.csv'
# test_split_pth = r'C:\Users\vian8\Desktop\Tugas2\LLIE\LoLI-Street Dataset\LoLI-Street Dataset\test_split.csv'

val_split_pth = r'C:\Users\Sistem Cerdas Five\Desktop\vian\afre\lesgo\New folder\val_split.csv'
test_split_pth = r'C:\Users\Sistem Cerdas Five\Desktop\vian\afre\lesgo\New folder\test_split.csv'

# 1. Load the original 200-row CSV
df_val_full = pd.read_csv(val_curated_pth)

# 2. Shuffle the rows 
# Using random_state=42 ensures you get the exact same split every time you run this script
df_shuffled = df_val_full.sample(frac=1, random_state=42).reset_index(drop=True)

# 3. Split down the middle (100 and 100)
df_val_final = df_shuffled.iloc[:100]
df_test_final = df_shuffled.iloc[100:]

# 4. Save the new splits
df_val_final.to_csv(val_split_pth, index=False)
df_test_final.to_csv(test_split_pth, index=False)

print(f"Created Val Split: {len(df_val_final)} scenes")
print(f"Created Test Split: {len(df_test_final)} scenes")

try:
    data_transform = T.ToTensor() 

    print("Initializing Train Dataset...")
    train_dataset = LoLiStreetDataset_Curated(
        csv_path=train_curated_pth,
        high_dir=os.path.join(train_root, 'high'), 
        low_dir=os.path.join(train_root, 'low'), 
        high_gt_dir=os.path.join(train_GT_root, 'high/Labels'),
        low_gt_dir=os.path.join(train_GT_root, 'low/Labels')
    )
    
    print("Initializing Val Dataset...")
    val_dataset = LoLiStreetDataset_Curated(
        csv_path=val_split_pth,
        high_dir=os.path.join(val_root, 'high'), 
        low_dir=os.path.join(val_root, 'low'), 
        high_gt_dir=os.path.join(val_GT_root, 'YOLO Annotations (high)/Labels'),
        low_gt_dir=os.path.join(val_GT_root, 'YOLO Annotations (low)/Labels')
    )

    print("Initializing Val Dataset...")
    test_dataset = LoLiStreetDataset_Curated(
        csv_path=test_split_pth,
        high_dir=os.path.join(val_root, 'high'), 
        low_dir=os.path.join(val_root, 'low'), 
        high_gt_dir=os.path.join(val_GT_root, 'YOLO Annotations (high)/Labels'),
        low_gt_dir=os.path.join(val_GT_root, 'YOLO Annotations (low)/Labels')
    )
    
    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=LoLiStreetDataset_Curated.collate_fn, num_workers=0, pin_memory=True)#, persistent_workers=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=LoLiStreetDataset_Curated.collate_fn, num_workers=0, pin_memory=True)#, persistent_workers=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=LoLiStreetDataset_Curated.collate_fn, num_workers=0, pin_memory=True)#, persistent_workers=True)

    print(f"\nSUCCESS: Datasets created. Train size: {len(train_dataset)}")

except Exception as e:
    print(f"\n[ERROR] Failed to initialize datasets: {e}")

Created Val Split: 100 scenes
Created Test Split: 100 scenes
Initializing Train Dataset...
Parsing CSV... Found 1700 curated scenes.
Dataset Built! Expanded to 5100 total image-label pairs.
Initializing Val Dataset...
Parsing CSV... Found 100 curated scenes.
Dataset Built! Expanded to 300 total image-label pairs.
Initializing Val Dataset...
Parsing CSV... Found 100 curated scenes.
Dataset Built! Expanded to 300 total image-label pairs.

SUCCESS: Datasets created. Train size: 5100


In [3]:
from training.train import YoloTrainer
from models.illuminet import YOLOv8_Clone
from losses.yolo_loss import DetectionLoss


yolo_clone = YOLOv8_Clone(back_use_arb=False)
det_loss_fn = DetectionLoss(model=yolo_clone, device=DEVICE)

yol_trainer = YoloTrainer(
    model=yolo_clone,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=60,
    det_loss_fn=det_loss_fn,
    experiment_name='YOLO_ONLY_BASE',
    device=DEVICE
)

yol_trainer_hist = yol_trainer.fit()
yol_trainer_hist_df = pd.DataFrame(yol_trainer_hist)
yol_trainer_hist_df.to_csv('yolo_only_train.csv', index=False)

355
Initializing nano params
NO ARB in Backbone
Initializing nano params
Initializing nano params
✅ Loaded 162 layers into Backbone.
Missing keys: 0 | Unexpected keys: 0
✅ Loaded 108 layers into Neck.
Missing keys: 0 | Unexpected keys: 0
✅ Loaded 85 layers into Head.
Missing keys: 0 | Unexpected keys: 0
Starting YOLO 2-Stage Training on cuda for 60 epochs...


Epoch 0/60:   0%|          | 0/638 [00:00<?, ?it/s]c:\Users\Sistem Cerdas Five\Desktop\vian\afre\lesgo\New folder\losses\yolo_loss.py:430: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  anchor_points, stride_tensor = make_anchors(preds, torch.tensor(self.stride, device=self.device))
c:\Users\Sistem Cerdas Five\Desktop\vian\afre\venv\Lib\site-packages\torch\functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\TensorShape.cpp:3596.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
c:\Users\Sistem Cerdas Five\Desktop\vian\afre\venv\Lib\site-packages\torch\optim\lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `o


🏆 **mAP Improved!** (0.0191 > -inf) at Epoch 0. Saving model.

--- Epoch 0 Summary (60.25s) ---
Train Loss: 81.5439 | Val Loss: 60.0059
Detection: mAP@0.5=0.019 | mAP=0.019
-------------------------------------------



Epoch 1/60: 100%|██████████| 638/638 [00:59<00:00, 10.70it/s, DLoss=17.2920, LR_BB=2.80e-05, LR_NH=2.80e-05]



🏆 **mAP Improved!** (0.1307 > 0.0191) at Epoch 1. Saving model.

--- Epoch 1 Summary (59.61s) ---
Train Loss: 47.0243 | Val Loss: 37.2676
Detection: mAP@0.5=0.131 | mAP=0.131
-------------------------------------------



...


No improvement in mAP (0.2680 ≤ 0.3118). Skipping save.

--- Epoch 57 Summary (59.14s) ---
Train Loss: 8.8659 | Val Loss: 22.7158
Detection: mAP@0.5=0.268 | mAP=0.268
-------------------------------------------



Epoch 58/60: 100%|██████████| 638/638 [00:58<00:00, 10.87it/s, DLoss=4.3925, LR_BB=8.47e-08, LR_NH=8.47e-08]



No improvement in mAP (0.2696 ≤ 0.3118). Skipping save.

--- Epoch 58 Summary (58.69s) ---
Train Loss: 8.8605 | Val Loss: 22.7610
Detection: mAP@0.5=0.270 | mAP=0.270
-------------------------------------------



Epoch 59/60: 100%|██████████| 638/638 [00:58<00:00, 10.96it/s, DLoss=4.0741, LR_BB=4.00e-10, LR_NH=4.00e-10]



No improvement in mAP (0.2700 ≤ 0.3118). Skipping save.

--- Epoch 59 Summary (58.23s) ---
Train Loss: 8.8566 | Val Loss: 22.7364
Detection: mAP@0.5=0.270 | mAP=0.270
-------------------------------------------

Training Complete. Final model saved to final_YOLO_ONLY_BASE_epoch60.pt


In [4]:
def visualize_train_plots(training_hist): # Extract data
    epochs = [h['epoch'] for h in training_hist]
    train_loss = [h['train_loss_total'] for h in training_hist]
    val_loss = [h['val_loss_total'] for h in training_hist]

    val_map50 = [h['val_mAP50'] for h in training_hist]
    val_prec_prox = [h['val_prec_prox'] for h in training_hist]
    val_recall = [h['val_recall'] for h in training_hist]

    val_psnr = [h['val_psnr'] for h in training_hist]
    val_ssim = [h['val_ssim'] for h in training_hist]

    # Set up the figure with 3 subplots
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('IllumiNet Multi-Task Training Progress', fontsize=16, fontweight='bold')

    # --- Plot 1: Total Loss ---
    axs[0].plot(epochs, train_loss, label='Train Loss', color='tab:blue', marker='o', linewidth=2)
    axs[0].plot(epochs, val_loss, label='Validation Loss', color='tab:orange', marker='s', linewidth=2)
    axs[0].set_title('Total Loss Convergence')
    axs[0].set_xlabel('Epoch')
    axs[0].set_ylabel('Loss')
    axs[0].set_xticks(epochs)
    axs[0].grid(True, linestyle='--', alpha=0.6)
    axs[0].legend()

    # --- Plot 2: Detection Metrics ---
    axs[1].plot(epochs, val_map50, label='mAP@0.5', color='tab:green', marker='o', linewidth=2)
    axs[1].plot(epochs, val_prec_prox, label='Precision', color='tab:red', marker='^', linewidth=2) # Added Precision
    axs[1].plot(epochs, val_recall, label='Recall', color='tab:purple', marker='s', linewidth=2)
    axs[1].set_title('YOLO Detection Performance')
    axs[1].set_xlabel('Epoch')
    axs[1].set_ylabel('Score')
    axs[1].set_xticks(epochs)
    axs[1].grid(True, linestyle='--', alpha=0.6)
    axs[1].legend()

    # --- Plot 3: Enhancement Metrics (Dual Axis) ---
    ax3 = axs[2]
    ax3_twin = ax3.twinx()

    line1 = ax3.plot(epochs, val_psnr, label='PSNR (dB)', color='tab:red', marker='o', linewidth=2)
    line2 = ax3_twin.plot(epochs, val_ssim, label='SSIM', color='tab:brown', marker='s', linewidth=2)

    ax3.set_title('LLIE Enhancement Performance')
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('PSNR (dB)', color='tab:red')
    ax3_twin.set_ylabel('SSIM', color='tab:brown')
    ax3.set_xticks(epochs)
    ax3.grid(True, linestyle='--', alpha=0.6)

    # Combine legends for the dual axis
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax3.legend(lines, labels, loc='lower right')

    plt.tight_layout()
    plt.show()
    


visualize_train_plots(training_hist=yol_trainer_hist)


KeyError: 'train_loss_total'

In [5]:
import os
import torch
import pandas as pd
from tqdm import tqdm
from torchvision.ops import nms
from torchmetrics.detection.mean_ap import MeanAveragePrecision

class YOLOEvaluator:
    def __init__(self, model, dataloader, device=None, min_area=None):
        self.model = model
        self.dataloader = dataloader
        self.device = device if device else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.min_area = min_area
        
        self.model.to(self.device)
        self.model.eval()
        
        # Tracking for Detection (Strictly @0.5 IoU, class metrics ON)
        metric_kwargs = {
            'box_format': 'xyxy',
            'iou_type': 'bbox',
            'iou_thresholds': [0.5],
            'max_detection_thresholds': [10, 100, 500],
            'class_metrics': True
        }
        
        self.map_metrics = {
            'light':    MeanAveragePrecision(**metric_kwargs).to(self.device),
            'moderate': MeanAveragePrecision(**metric_kwargs).to(self.device),
            'dense':    MeanAveragePrecision(**metric_kwargs).to(self.device),
            'total':    MeanAveragePrecision(**metric_kwargs).to(self.device)
        }

    def _get_best_epoch(self, hist_csv_path):
        if not hist_csv_path or not os.path.exists(hist_csv_path): return "N/A"
        try:
            df = pd.read_csv(hist_csv_path)
            target_col = 'val_mAP50' if 'val_mAP50' in df.columns else 'val_mAP'
            if target_col in df.columns:
                return int(df.loc[df[target_col].idxmax(), 'epoch'])
        except Exception as e:
            print(f"Could not read best epoch from {hist_csv_path}: {e}")
        return "N/A"

    def _get_class_map(self, map_res, target_class_id=2):
        """Extracts mAP for Car (Class 2)"""
        if map_res.get('map_per_class') is None or len(map_res['map_per_class']) == 0: return 0.0
        classes = map_res['classes'].tolist()
        if target_class_id in classes:
            val = map_res['map_per_class'][classes.index(target_class_id)].item()
            return val if val != -1.0 else 0.0
        return 0.0

    def evaluate(self, weight_path, model_name="Baseline YOLO", hist_csv_path=None):
        print(f"\nLoading weights for {model_name} from: {weight_path}")
        self.model.load_state_dict(torch.load(weight_path, map_location=self.device))

        # Reset Trackers
        for illum in ['light', 'moderate', 'dense', 'total']:
            self.map_metrics[illum].reset()

        if self.min_area:
            print(f"🛑 FILTER ACTIVE: Ignoring objects smaller than {self.min_area} pixels!")

        with torch.no_grad():
            for batch in tqdm(self.dataloader, desc=f"Evaluating {model_name}"):
                # 1. Load Data directly from your dataloader
                low_imgs = batch['images_low'].to(self.device)
                im_ids = batch['im_id'] 
                B, _, H, W = low_imgs.shape

                # 2. Forward Pass
                with torch.autocast(device_type=self.device.type, enabled=(self.device.type == 'cuda')):
                    det_out = self.model(low_imgs, inference=True)
                
                # 3. Process Batch Iteratively
                for b in range(B):
                    # Identify illumination type from filename
                    illum = 'total'
                    for i_type in ['light', 'moderate', 'dense']:
                        if i_type in im_ids[b].lower():
                            illum = i_type
                            break

                    # --- DETECTION LOGIC ---
                    p = det_out[b].permute(1, 0).contiguous()
                    boxes_xywh = p[:, :4]
                    confidences, class_indices = torch.max(p[:, 4:], dim=1)
                    
                    pred_xy = boxes_xywh[:, :2]
                    pred_wh = boxes_xywh[:, 2:]
                    boxes_xyxy = torch.cat((pred_xy - pred_wh / 2, pred_xy + pred_wh / 2), dim=1)
                    
                    mask = confidences > 0.001 
                    p_boxes, p_scores, p_labels = boxes_xyxy[mask], confidences[mask], class_indices[mask]
                    
                    if len(p_boxes) > 0:
                        keep = nms(p_boxes, p_scores, iou_threshold=0.45)
                        p_boxes, p_scores, p_labels = p_boxes[keep], p_scores[keep], p_labels[keep]
                    
                    # --- GT Extraction (Scaled back to absolute dimensions for metrics) ---
                    gt_mask = batch['batch_idx'] == b
                    gt_boxes_xywh = batch['bboxes'][gt_mask].to(self.device)
                    gt_labels = batch['cls'][gt_mask].to(self.device)

                    if len(gt_boxes_xywh) > 0:
                        gt_xy = gt_boxes_xywh[:, :2]
                        gt_wh = gt_boxes_xywh[:, 2:]
                        gt_boxes_xyxy = torch.cat((gt_xy - gt_wh / 2, gt_xy + gt_wh / 2), dim=1)
                        gt_boxes_xyxy[:, [0, 2]] *= W
                        gt_boxes_xyxy[:, [1, 3]] *= H
                    else:
                        gt_boxes_xyxy = torch.empty((0, 4), device=self.device)

                    # --- SIZE FILTER ---
                    if self.min_area is not None:
                        if len(gt_boxes_xyxy) > 0:
                            gt_keep = ((gt_boxes_xyxy[:, 2] - gt_boxes_xyxy[:, 0]) * (gt_boxes_xyxy[:, 3] - gt_boxes_xyxy[:, 1])) >= self.min_area
                            gt_boxes_xyxy, gt_labels = gt_boxes_xyxy[gt_keep], gt_labels[gt_keep]
                        if len(p_boxes) > 0:
                            p_keep = ((p_boxes[:, 2] - p_boxes[:, 0]) * (p_boxes[:, 3] - p_boxes[:, 1])) >= self.min_area
                            p_boxes, p_scores, p_labels = p_boxes[p_keep], p_scores[p_keep], p_labels[p_keep]
                    
                    pred_dict = {"boxes": p_boxes, "scores": p_scores, "labels": p_labels}
                    gt_dict = {"boxes": gt_boxes_xyxy, "labels": gt_labels}
                    
                    self.map_metrics[illum].update([pred_dict], [gt_dict])
                    self.map_metrics['total'].update([pred_dict], [gt_dict])

        # --- COMPUTE AVERAGES ---
        res_total = self.map_metrics['total'].compute()
        res_light = self.map_metrics['light'].compute()
        res_mod = self.map_metrics['moderate'].compute()
        res_dense = self.map_metrics['dense'].compute()

        def fix_map(val):
            v = val.item() if torch.is_tensor(val) else val
            return v if v != -1.0 else 0.0

        summary = {
            'Weights': os.path.basename(weight_path),
            'mAP50_Total': fix_map(res_total['map_50']),
            'mAP50_Small': fix_map(res_total['map_small']),
            'mAP50_Medium': fix_map(res_total['map_medium']),
            'mAP50_Large': fix_map(res_total['map_large']),
            
            'mAP_Light': fix_map(res_light['map_50']),
            'mAP_Moderate': fix_map(res_mod['map_50']),
            'mAP_Dense': fix_map(res_dense['map_50']),
            
            'mAP_Car_Total': self._get_class_map(res_total, target_class_id=2),
            'mAP_Car_Light': self._get_class_map(res_light, target_class_id=2),
            'mAP_Car_Moderate': self._get_class_map(res_mod, target_class_id=2),
            'mAP_Car_Dense': self._get_class_map(res_dense, target_class_id=2),
            
            'Best_Epoch': self._get_best_epoch(hist_csv_path)
        }

        print("\n" + "="*50)
        print(" FINAL ROW RESULTS ")
        print("="*50)
        for key, value in summary.items():
            if isinstance(value, float): print(f"{key:>20}: {value:.4f}")
            else: print(f"{key:>20}: {value}")
                
        return summary

In [8]:
import pandas as pd

# ==========================================
# 1. DEFINE BASELINE YOLO CONFIGURATIONS
# ==========================================
baseline_yolo_configs = [
    # {
    #     'name': 'Baseline_YOLO_BackARB',
    #     'use_arb': True, # Evaluates the ARB-enabled YOLO on dark images
    #     'hist_csv': 'training_hist_Baseline_YOLO_BackARB.csv',
    #     'best_weights': 'best_Baseline_YOLO_BackARB_mAP.pt',
    #     'final_weights': 'final_Baseline_YOLO_BackARB_epoch60.pt'
    # },
    {
        'name': 'Baseline_YOLO_Vanilla',
        'use_arb': False, # Evaluates standard YOLO on dark images
        'hist_csv': r'C:\Users\Sistem Cerdas Five\Desktop\vian\afre\lesgo\New folder\yolo_only_train.csv',
        'best_weights': r'C:\Users\Sistem Cerdas Five\Desktop\vian\afre\lesgo\New folder\yolov8n.pt', #r'C:\Users\Sistem Cerdas Five\Desktop\vian\afre\lesgo\New folder\best_YOLO_ONLY_BASE_mAP.pt',
        'final_weights': r'C:\Users\Sistem Cerdas Five\Desktop\vian\afre\lesgo\New folder\final_YOLO_ONLY_BASE_epoch60.pt'
    }
]

results_list_baseline = []

# ==========================================
# 2. RUN EVALUATION
# ==========================================
for cfg in baseline_yolo_configs:
    print("\n" + "="*60)
    print(f" EVALUATING BASELINE CONTROL: {cfg['name']} ")
    print("="*60)
    
    yolo_clone = YOLOv8_Clone(back_use_arb=cfg['use_arb'])
    
    # Pass your dataloader that points to the raw dark images
    evaluator = YOLOEvaluator(
        model=yolo_clone, 
        dataloader=test_loader, # <--- Replace with your raw low-light test loader
        device=DEVICE
    )
    
    if os.path.exists(cfg['best_weights']):
        display_name = f"{cfg['name']} (Best)"
        res_best = evaluator.evaluate(weight_path=cfg['best_weights'], model_name=display_name, hist_csv_path=cfg['hist_csv'])
        if res_best:
            res_best['Model'] = display_name
            results_list_baseline.append(res_best)

    if os.path.exists(cfg['final_weights']):
        display_name = f"{cfg['name']} (Final)"
        res_final = evaluator.evaluate(weight_path=cfg['final_weights'], model_name=display_name, hist_csv_path=cfg['hist_csv'])
        if res_final:
            res_final['Model'] = display_name
            results_list_baseline.append(res_final)

# ==========================================
# 3. EXPORT RESULTS
# ==========================================
if results_list_baseline:
    df_baseline = pd.DataFrame(results_list_baseline)
    cols = ['Model'] + [c for c in df_baseline.columns if c != 'Model']
    df_baseline = df_baseline[cols]
    
    print("\n" + "="*80)
    print(" BASELINE YOLO RESULTS TABLE ")
    print("="*80)
    print(df_baseline.to_string(index=False))
    
    df_baseline.to_csv("Control_Baseline_Metrics.csv", index=False)


 EVALUATING BASELINE CONTROL: Baseline_YOLO_Vanilla 
Initializing nano params
NO ARB in Backbone
Initializing nano params
Initializing nano params
✅ Loaded 162 layers into Backbone.
Missing keys: 0 | Unexpected keys: 0
✅ Loaded 108 layers into Neck.
Missing keys: 0 | Unexpected keys: 0
✅ Loaded 85 layers into Head.
Missing keys: 0 | Unexpected keys: 0

Loading weights for Baseline_YOLO_Vanilla (Best) from: C:\Users\Sistem Cerdas Five\Desktop\vian\afre\lesgo\New folder\yolov8n.pt


C:\Users\Sistem Cerdas Five\AppData\Local\Temp\ipykernel_24572\288482213.py:56: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(torch.load(weight_pa

RuntimeError: Error(s) in loading state_dict for YOLOv8_Clone:
	Missing key(s) in state_dict: "backbone.conv0.conv.weight", "backbone.conv0.bn.weight", "backbone.conv0.bn.bias", "backbone.conv0.bn.running_mean", "backbone.conv0.bn.running_var", "backbone.conv1.conv.weight", "backbone.conv1.bn.weight", "backbone.conv1.bn.bias", "backbone.conv1.bn.running_mean", "backbone.conv1.bn.running_var", "backbone.conv3.conv.weight", "backbone.conv3.bn.weight", "backbone.conv3.bn.bias", "backbone.conv3.bn.running_mean", "backbone.conv3.bn.running_var", "backbone.conv5.conv.weight", "backbone.conv5.bn.weight", "backbone.conv5.bn.bias", "backbone.conv5.bn.running_mean", "backbone.conv5.bn.running_var", "backbone.conv7.conv.weight", "backbone.conv7.bn.weight", "backbone.conv7.bn.bias", "backbone.conv7.bn.running_mean", "backbone.conv7.bn.running_var", "backbone.c2f_2.conv1.conv.weight", "backbone.c2f_2.conv1.bn.weight", "backbone.c2f_2.conv1.bn.bias", "backbone.c2f_2.conv1.bn.running_mean", "backbone